# SE374.R11 - Scientific Research & Benchmark Runner
## 5 Chuyen De Nghiem Thu Khoa Hoc (Research-Grade, mode-aware)

* **Mon hoc:** SE374.R11 - Phat trien phan mem xu ly anh y khoa (UIT - DHQG-HCM)
* **Giang vien huong dan:** TS. Bui Cao Doanh
* **Nhom thuc hien:** [Nhom de xuat rieng - Ho va ten Sinh vien (MSSV)]
* **Kien truc:** **GitHub-First Standalone Scientific Runner** - Tu dong clone ma nguon truc tiep tu GitHub repository (`https://github.com/khoinguyen59/medical-wsss.git`), khong can ket noi Google Drive. Toan bo ket qua nghien cuu duoc tu dong nen va tai truc tiep ve may tinh nguoi dung.
* **Che do chay mac dinh:** `SANITY_DRY_RUN = True` (mode `SANITY`: 50 Val / 50 Test / 20 LUAD / 3 Whole-ROI) de kiem tra pipeline. Chi khi dat `SANITY_DRY_RUN = False` moi la chay `FULL`. Ten goi ZIP, `RUN_META` va `provenance` trong summary luon ghi kem mode, nen khong the nham ket qua sanity voi ket qua day du.
* **5 Thi Nghiem Khoa Hoc:**
  1. **Validation-to-Test Scientific Protocol**: Quet sieu tham so tren Val, dong bang va danh gia tren Test (mIoU, Dice, FwIoU, Pixel Accuracy, Boundary F1), chia tap theo Patient ID.
  2. **Multi-Magnification & Coarse-to-Fine Adaptive Router**: Mo phong phan giai da tang (5x, 10x, 20x, 40x), router do phuc tap thong minh, duong cong Pareto Adaptive vs Random. Moi so lieu tiet kiem tinh toan deu do tai runtime, khong hardcode.
  3. **Patient-Level Clinical Biomarker Correlation**: Tinh toan VTP, TSR theo cap do benh nhan, Pearson r, Spearman rho, Bland-Altman va do phu hop nguy co CAP, kem sample size that (so patch dung / bo qua / so benh nhan).
  4. **In-Domain LUAD + BCSS Transfer Baseline**: Thi nghiem 4 KHONG phai zero-shot. Checkpoint nap qua `ensure_luad_checkpoints` va `load_luad_models` la checkpoint LUAD in-domain, ket qua in duoi nhan `[IN-DOMAIN LUAD]`. Ket qua chuyen trong so BCSS (Breast) sang LUAD (Lung) do `run_cross_dataset_luad_benchmark` tra ve duoi khoa `zero_shot_bcss_to_luad_transfer`, in rieng duoi nhan `[TRANSFER BCSS -> LUAD]`. Hai ket qua khac nhau ve ban chat va khong duoc gop lam mot.
  5. **Whole-ROI Evaluation (coverage reported)**: Danh gia tren tap Whole-ROI BCSS doi sanh Ground-Truth chuyen gia, kem `coverage_ratio` va co `is_partial_coverage`; khi coverage < 100% thi `metric_valid = False`.

---
## Buoc 1: Dong Bo Ma Nguon Truc Tiep Tu GitHub & Khoi Tao Moi Truong GPU (Zero Google Drive)
O code duoi day tu dong:
1. Clone ma nguon truc tiep tu GitHub: `https://github.com/khoinguyen59/medical-wsss.git` (hoac pull ban moi nhat neu da ton tai).
2. Chuyen vao thu muc du an va nap duong dan he thong vao `sys.path`.
3. Bao toan ABI C-extension cua NumPy/PyTorch/SciPy tren Colab (tu dong khoi phuc neu phien chay truoc bi lech NumPy ABI) va chi cai bo sung cac goi tai du lieu nhe (`gdown`, `huggingface_hub`, `tifffile`).
4. Khoi tao hat giong ngau nhien co dinh (seed 42) va xac thuc GPU (Hoan toan khong can mount Google Drive).

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/khoinguyen59/medical-wsss.git"
REPO_DIR = Path("/content/medical-wsss") if IN_COLAB else Path.cwd().resolve()

# RUN_WARNINGS: moi canh bao ha tang duoc ghi lai va dua vao provenance cua summary (Cell 18).
RUN_WARNINGS = []
GIT_SYNC_STATUS = {"clone": "skipped", "fetch": "skipped", "reset": "skipped"}

if IN_COLAB:
    if not (REPO_DIR / "src" / "pipeline").is_dir():
        print(f"--> Dang clone ma nguon truc tiep tu GitHub: {REPO_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
        GIT_SYNC_STATUS["clone"] = "ok"
    else:
        print("--> Cap nhat ma nguon moi nhat tu GitHub...")
        fetch_proc = subprocess.run(
            ["git", "-C", str(REPO_DIR), "fetch", "origin", "master"],
            check=False, capture_output=True, text=True,
        )
        GIT_SYNC_STATUS["fetch"] = f"returncode={fetch_proc.returncode}"
        if fetch_proc.returncode != 0:
            _fetch_err = (fetch_proc.stderr or "").strip().splitlines()[-1:] or ["(khong co stderr)"]
            RUN_WARNINGS.append(f"git fetch that bai (returncode={fetch_proc.returncode}): {_fetch_err[0][:300]}")
            print(f"[CANH BAO] git fetch that bai (returncode={fetch_proc.returncode}): {_fetch_err[0][:300]}")
            print("[CANH BAO] Tiep tuc voi ma nguon hien co; KHONG dam bao day la ban moi nhat tren origin/master.")
        reset_proc = subprocess.run(
            ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/master"],
            check=False, capture_output=True, text=True,
        )
        GIT_SYNC_STATUS["reset"] = f"returncode={reset_proc.returncode}"
        if reset_proc.returncode != 0:
            _reset_err = (reset_proc.stderr or "").strip().splitlines()[-1:] or ["(khong co stderr)"]
            RUN_WARNINGS.append(f"git reset --hard that bai (returncode={reset_proc.returncode}): {_reset_err[0][:300]}")
            print(f"[CANH BAO] git reset --hard origin/master that bai (returncode={reset_proc.returncode}): {_reset_err[0][:300]}")
            print("[CANH BAO] Ma nguon dang chay co the la ban cu hoac bi tron. Hay clone lai truoc khi bao cao ket qua.")

    os.chdir(REPO_DIR)

    # Self-heal if a previous cell execution downgraded NumPy on a NumPy 2.x Colab kernel.
    # Chi bat cac loi ABI/dependency cu the; loi khac phai noi len thay vi bi che di.
    try:
        import numpy as np
        import numpy.random
    except (ImportError, ModuleNotFoundError, ValueError, AttributeError) as np_abi_err:
        print(f"--> Phat hien xung dot ABI NumPy tu phien truoc ({type(np_abi_err).__name__}), dang khoi phuc NumPy 2.x tuong thich Colab...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "--no-cache-dir", "--upgrade", "numpy>=2.0.0"], check=False)
        for _k in [k for k in list(sys.modules.keys()) if k == "numpy" or k.startswith("numpy.")]:
            sys.modules.pop(_k, None)
        import numpy as np
        import numpy.random

    missing_pkgs = [
        pkg for mod, pkg in [
            ("gdown", "gdown"),
            ("huggingface_hub", "huggingface_hub"),
            ("tifffile", "tifffile"),
        ]
        if importlib.util.find_spec(mod) is None
    ]
    if missing_pkgs:
        print(f"--> Dang cai dat goi bo tro con thieu: {missing_pkgs}...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "--no-cache-dir", "--no-deps"] + missing_pkgs, check=False)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--prefer-binary", "--no-cache-dir"] + missing_pkgs, check=False)
    for _mod in [k for k in list(sys.modules.keys()) if k == "src" or k.startswith("src.")]:
        sys.modules.pop(_mod, None)

CODE_ROOT = str(REPO_DIR)
if CODE_ROOT not in sys.path:
    sys.path.insert(0, CODE_ROOT)

from src.pipeline.colab_runner import set_reproducible_seed
ENV_INFO = set_reproducible_seed(seed=42, verbose=True)
print(f"[READY] CODE_ROOT = {CODE_ROOT}")
print(f"[READY] Device = {ENV_INFO['device']} | CUDA Available = {ENV_INFO['cuda_available']}")
if RUN_WARNINGS:
    print(f"[READY] {len(RUN_WARNINGS)} canh bao ha tang da duoc ghi vao RUN_WARNINGS (xem Cell 18).")

---
## Buoc 2: Thiet Lap Cau Hinh & Che Do Kiem Dinh (Sanity Dry-Run Toggle)

> **Chu y ve tai nguyen GPU:**
- `SANITY_DRY_RUN = True`: Chay thu nghiem mau nhanh (50 Val, 50 Test, 20 LUAD, 1 Whole-ROI, ~2 phut) de kiem tra tinh dung dan toan bo pipeline truoc khi chay day du.
- `SANITY_DRY_RUN = False`: Chay thuc nghiem toan dien tren 100% du lieu thuc te.

In [ ]:
import json
import math
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

SANITY_DRY_RUN = True
DRY_RUN_SAMPLES = 50
USE_AMP = True
AMP_DTYPE = torch.float16

RUN_MODE = "SANITY" if SANITY_DRY_RUN else "FULL"
_run_now = datetime.now(timezone.utc)
RUN_TIMESTAMP = _run_now.strftime("%Y%m%dT%H%M%SZ")
RUN_STARTED_UTC = _run_now.strftime("%Y-%m-%dT%H:%M:%SZ")


def _read_git_commit(repo_dir):
    """Doc commit hash that bang subprocess; tra ve 'unknown' neu khong doc duoc (khong doan)."""
    try:
        proc = subprocess.run(
            ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
            check=False, capture_output=True, text=True,
        )
    except (OSError, ValueError):
        return "unknown"
    if proc.returncode != 0:
        return "unknown"
    return (proc.stdout or "").strip() or "unknown"


_repo_for_commit = Path(globals().get("REPO_DIR", Path.cwd())).resolve()
_work_root_base = Path("/content") if IN_COLAB else (_repo_for_commit / "se374_scientific_work")
WORK_ROOT = Path(_work_root_base).resolve()
# Moi lan chay co thu muc rieng: khong tron JSON cua lan truoc vao validation/ZIP.
OUTPUT_DIR = WORK_ROOT / "scientific_results" / f"{RUN_MODE}_{RUN_TIMESTAMP}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_META = {
    "run_mode": RUN_MODE,
    "sanity_dry_run": bool(SANITY_DRY_RUN),
    "dry_run_samples": int(DRY_RUN_SAMPLES),
    "seed": 42,
    "git_commit": _read_git_commit(_repo_for_commit),
    "notebook_path": "notebooks/SE374_Colab_Scientific_Research_Runner_FIXED.ipynb",
    "run_started_utc": RUN_STARTED_UTC,
    "run_timestamp": RUN_TIMESTAMP,
    "work_root": str(WORK_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "cwd": str(Path.cwd().resolve()),
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "device": str(ENV_INFO.get("device", "cpu")),
    "gpu_name": str(ENV_INFO.get("gpu_name", "CPU")),
    "gpu_vram_gb": round(float(ENV_INFO.get("vram_gb", 0.0) or 0.0), 2),
    "amp": bool(USE_AMP),
    "amp_dtype": str(AMP_DTYPE),
    "git_sync_status": dict(globals().get("GIT_SYNC_STATUS", {})),
    "run_warnings": list(globals().get("RUN_WARNINGS", [])),
    "split_provenance": {"source": "pending"},
    "sample_sizes": {},
    "hyperparameters_locked": {},
}

# So do that cua tung thi nghiem (patch / benh nhan / slide) duoc dien dan boi cac cell sau.
RUN_ERRORS = []
EXPERIMENT_STATUS = {
    "exp1_val_to_test": "not_run",
    "exp2_multimag_pareto": "not_run",
    "exp3_biomarker": "not_run",
    "exp4_luad_in_domain": "not_run",
    "exp4b_bcss_to_luad_transfer": "not_run",
    "exp5_whole_roi": "not_run",
    "export_summary": "not_run",
}

# Ket qua mac dinh la None: Cell 18 khong dung locals() nua, chi doc cac bien da khai bao.
val_to_test_results = None
pareto_data = None
pareto_export = None
bio_results = None
luad_results = None
luad_transfer_results = None
roi_metrics = None
SETUP_ERROR = None


def record_error(experiment, exc, stage=""):
    """Ghi loi that bai vao RUN_ERRORS (khong nuot loi) va in ra de nguoi doc thay."""
    entry = {
        "experiment": str(experiment),
        "stage": str(stage),
        "type": type(exc).__name__,
        "message": str(exc)[:800],
    }
    RUN_ERRORS.append(entry)
    _suffix = f"/{stage}" if stage else ""
    print(f"[FAIL] {experiment}{_suffix}: {type(exc).__name__}: {str(exc)[:400]}")
    return entry


def nan_to_none(obj, path="root", found=None):
    """Doi NaN/Inf thanh None va ghi duong dan; KHONG doi thanh 0.0.

    Xu ly ca numpy floating (np.float32) vi isinstance(np.float32(0.0), float) la False.
    """
    if found is None:
        found = []
    if isinstance(obj, (float, np.floating)):
        _val = float(obj)
        if math.isnan(_val) or math.isinf(_val):
            found.append(path)
            return None
        return _val
    if isinstance(obj, dict):
        return {k: nan_to_none(v, f"{path}.{k}", found) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [nan_to_none(v, f"{path}[{i}]", found) for i, v in enumerate(obj)]
    return obj


def write_json_atomic(path, text):
    """Ghi JSON bang temp + replace; khong de lai file 0 byte khi ghi loi."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    try:
        tmp.write_text(text, encoding="utf-8")
        tmp.replace(path)
    finally:
        try:
            if tmp.exists():
                tmp.unlink()
        except OSError:
            pass


_gpu_name = RUN_META["gpu_name"]
_gpu_vram = RUN_META["gpu_vram_gb"]
_run_commit = RUN_META["git_commit"]
_run_seed = RUN_META["seed"]
_run_started = RUN_META["run_started_utc"]

print(f"[CONFIG] RUN_MODE = {RUN_MODE} | SANITY_DRY_RUN = {SANITY_DRY_RUN} | Samples per split = {DRY_RUN_SAMPLES}")
print(f"[CONFIG] Device: {ENV_INFO['device']} | AMP: {USE_AMP} ({AMP_DTYPE})")
print(f"[CONFIG] GPU: {_gpu_name} | VRAM: {_gpu_vram:.2f} GB")
print("[CONFIG] GPU tier: notebook khong tu doan tier; doi chieu ten GPU o tren voi goi ban dang dung.")
print(f"[CONFIG] WORK_ROOT (absolute): {WORK_ROOT}")
print(f"[CONFIG] Output Directory (run-specific): {OUTPUT_DIR}")
print(f"[CONFIG] Provenance: commit={_run_commit} | seed={_run_seed} | mode={RUN_MODE} | started={_run_started}")
if RUN_META["run_warnings"]:
    print(f"[CANH BAO] {len(RUN_META['run_warnings'])} canh bao ha tang nam trong RUN_META['run_warnings'].")

---
## Buoc 3: Tai Du Lieu BCSS & Nap Cac Mo Hinh Pretrained (`Stage-1`, `Stage-2`, `Engine`)
Tu dong tai dataset BCSS va ca hai checkpoint chinh thuc (`stage1_checkpoint.pth` va `stage2_checkpoint.pth`).

In [ ]:
from src.colab_pkg.downloader import ensure_bcss_dataset, ensure_checkpoints, resolve_split_dirs
from src.colab_pkg.model_loader import ColabModelLoader, MultiMethodWSSSEngineV2

DATA_ROOT = None
TEST_IMG_DIR = None
TEST_MASK_DIR = None
CKPT_PATHS = {}
MODEL_STAGE1 = None
MODEL_STAGE2 = None
WSSS_ENGINE = None

try:
    DATA_ROOT, TEST_IMG_DIR, TEST_MASK_DIR = ensure_bcss_dataset(
        data_root=WORK_ROOT / "BCSS-WSSS",
        zip_path=WORK_ROOT / "BCSS-WSSS.zip",
        extract_dir=WORK_ROOT,
        drive_cache_dir=None,
    )
    CKPT_PATHS = ensure_checkpoints(
        checkpoint_dir=WORK_ROOT / "checkpoints",
        drive_cache_dir=None,
        require_stage2=True,
    )
    loader = ColabModelLoader(device=ENV_INFO["device"])
    MODEL_STAGE1 = loader.load_stage1_netcam(CKPT_PATHS["stage1"])
    MODEL_STAGE2 = loader.load_stage2_deeplabv3plus(CKPT_PATHS["stage2"])
    WSSS_ENGINE = MultiMethodWSSSEngineV2(
        model_stage1=MODEL_STAGE1, use_amp=USE_AMP, amp_dtype=AMP_DTYPE,
    )
    print("[OK] Models loaded successfully: Stage-1 Net_CAM and Stage-2 DeepLabV3+!")
except Exception as _setup_err:
    SETUP_ERROR = _setup_err
    record_error("setup", _setup_err, "cell6_load_models")
    for _exp_key in list(EXPERIMENT_STATUS.keys()):
        EXPERIMENT_STATUS[_exp_key] = "blocked_by_setup"
    DATA_ROOT = WORK_ROOT / "_setup_failed_no_data"
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    TEST_IMG_DIR = None
    TEST_MASK_DIR = None
    print("[FAIL] Setup that bai; Cell 8-16 se ghi nhan va bo qua, Cell 18 van xuat summary/ZIP de giu bang chung.")

---
## Buoc 4: Thi Nghiem 1 - Quy Trinh Khoa Hoc Validation-to-Test (Zero Patient Leakage)
Quy trinh khoa hoc chuan:
1. Phan tach benh nhan giua Validation va Test triet de theo Patient ID (INVARIANT-08).
2. Quet va chon sieu tham so hau xu ly tren tap Validation (`val_files`).
3. Dong bang sieu tham so da chon va danh gia khach quan tren tap Test (`test_files`).
4. Tinh toan day du: `4-class mIoU`, `mDice`, `FwIoU`, `Pixel Accuracy`, va `Boundary F1`.

In [ ]:
from pathlib import Path
from collections import defaultdict
from src.colab_pkg.downloader import extract_patient_id, resolve_split_dirs
from src.colab_pkg.evaluator import run_val_to_test_scientific_protocol


def _group_by_patient(files):
    grouped = defaultdict(list)
    for _p in files:
        grouped[extract_patient_id(_p.name)].append(_p)
    return grouped


def _patches_in_dir(img_dir):
    if img_dir is None or not Path(img_dir).is_dir():
        return []
    return sorted(Path(img_dir).glob("*.png"))


_, test_img_dir, test_mask_dir = resolve_split_dirs(DATA_ROOT, split="test")
if test_img_dir is None or not test_img_dir.is_dir() or len(list(test_img_dir.glob("*.png"))) == 0:
    test_img_dir, test_mask_dir = TEST_IMG_DIR, TEST_MASK_DIR

_, val_img_dir, val_mask_dir = resolve_split_dirs(DATA_ROOT, split="val")

val_files = []
test_files = []
val_patient_map = {}
test_patient_map = {}

_official_val = _patches_in_dir(val_img_dir)
_official_test = _patches_in_dir(test_img_dir)
_use_official_split = (
    val_img_dir is not None
    and test_img_dir is not None
    and len(_official_val) > 0
    and len(_official_test) > 0
    and Path(val_img_dir).resolve() != Path(test_img_dir).resolve()
)

if _use_official_split:
    val_files = _official_val
    test_files = _official_test
    val_patient_map = dict(_group_by_patient(val_files))
    test_patient_map = dict(_group_by_patient(test_files))
    SPLIT_PROVENANCE = {
        "source": "official_val_test_dirs",
        "val_img_dir": str(val_img_dir),
        "test_img_dir": str(test_img_dir),
        "method": "patient_id_from_filename",
    }
else:
    # KHONG con nhanh chia theo chi so patch. Moi phep chia deu o cap Patient ID (INVARIANT-08).
    _all_test_patches = _official_test
    _patient_map = dict(_group_by_patient(_all_test_patches))
    _unique_pids = sorted(_patient_map.keys())
    if len(_unique_pids) >= 2:
        _n_val_pids = max(1, len(_unique_pids) // 3)
        _val_pids = sorted(_unique_pids[:_n_val_pids])
        _test_pids = sorted(_unique_pids[_n_val_pids:])
        val_patient_map = {_pid: _patient_map[_pid] for _pid in _val_pids}
        test_patient_map = {_pid: _patient_map[_pid] for _pid in _test_pids}
        val_files = [p for _pid in _val_pids for p in _patient_map[_pid]]
        test_files = [p for _pid in _test_pids for p in _patient_map[_pid]]
        val_mask_dir = test_mask_dir
        SPLIT_PROVENANCE = {
            "source": "patient_id_fallback_within_test_split",
            "test_img_dir": str(test_img_dir),
            "method": "patient_id_from_filename",
            "warning": "Val duoc tach ra tu chinh split test, khong phai cohort doc lap; generalization gap chi mang tinh tham khao.",
        }
        print(f"[ANTI-LEAKAGE] Chia theo Patient ID: {len(_val_pids)} benh nhan Val ({len(val_files)} anh) | {len(_test_pids)} benh nhan Test ({len(test_files)} anh)")
        RUN_WARNINGS.append("Val split duoc tach tu test split (patient-level fallback); can than khi dien giai generalization gap.")
    else:
        # Khong du 2 benh nhan: danh dau skipped, KHONG tu che split.
        val_files = []
        test_files = []
        val_patient_map = {}
        test_patient_map = {}
        val_mask_dir = test_mask_dir
        SPLIT_PROVENANCE = {
            "source": "skipped_no_val_split",
            "test_img_dir": str(test_img_dir),
            "patients_found": _unique_pids,
            "reason": "Khong du 2 Patient ID de tach val/test o cap benh nhan.",
        }
        EXPERIMENT_STATUS["exp1_val_to_test"] = "skipped_no_val_split"
        RUN_ERRORS.append({
            "experiment": "exp1_val_to_test",
            "stage": "split",
            "type": "InsufficientPatients",
            "message": f"Chi tim thay {len(_unique_pids)} Patient ID; khong the tach val/test theo benh nhan.",
        })
        print(f"[SKIPPED] Chi co {len(_unique_pids)} Patient ID -> khong tao split gia, EXP1 duoc danh dau skipped_no_val_split.")

val_pids = sorted(val_patient_map.keys())
test_pids = sorted(test_patient_map.keys())

try:
    assert set(val_pids).isdisjoint(set(test_pids)), (
        f"Patient leakage: {len(set(val_pids) & set(test_pids))} Patient ID xuat hien ca o val va test."
    )
    _split_leak_free = True
except AssertionError as _leak_err:
    _split_leak_free = False
    record_error("exp1_val_to_test", _leak_err, "assert_patient_disjoint")
    EXPERIMENT_STATUS["exp1_val_to_test"] = "failed_patient_overlap"

if SANITY_DRY_RUN and val_patient_map and test_patient_map:
    quota_val = max(1, DRY_RUN_SAMPLES // max(1, len(val_pids)))
    val_files = [p for pid in val_pids for p in val_patient_map[pid][:quota_val]][:DRY_RUN_SAMPLES]
    quota_test = max(1, DRY_RUN_SAMPLES // max(1, len(test_pids)))
    test_files = [p for pid in test_pids for p in test_patient_map[pid][:quota_test]][:DRY_RUN_SAMPLES]

RUN_META["split_provenance"] = dict(SPLIT_PROVENANCE)
RUN_META["split_provenance"].update({
    "n_val_patients": len(val_pids),
    "n_test_patients": len(test_pids),
    "n_val_patches_scheduled": len(val_files),
    "n_test_patches_scheduled": len(test_files),
    "sanity_subsample_applied": bool(SANITY_DRY_RUN),
    "leak_free": _split_leak_free,
})
RUN_META["sample_sizes"]["exp1_val_to_test"] = {
    "n_val_files": len(val_files),
    "n_test_files": len(test_files),
    "n_val_patients": len(val_pids),
    "n_test_patients": len(test_pids),
}

_can_run_exp1 = (
    _split_leak_free
    and len(val_files) > 0
    and len(test_files) > 0
    and EXPERIMENT_STATUS["exp1_val_to_test"] not in ("skipped_no_val_split", "failed_patient_overlap")
)

if SETUP_ERROR is not None:
    EXPERIMENT_STATUS["exp1_val_to_test"] = "blocked_by_setup"
    _can_run_exp1 = False
    print("[SKIPPED] Setup that bai; EXP1 khong chay.")

if _can_run_exp1:
    print(f"[EXP 1] Running Val-to-Test Protocol: {len(val_files)} Val, {len(test_files)} Test...")
    _val_names_before = [p.name for p in val_files]
    _test_names_before = [p.name for p in test_files]
    try:
        val_to_test_results = run_val_to_test_scientific_protocol(
            engine=WSSS_ENGINE,
            model_stage2=MODEL_STAGE2,
            val_files=val_files,
            test_files=test_files,
            val_mask_dir=val_mask_dir,
            test_mask_dir=test_mask_dir,
            method_key="M6_MultiScale_Pyramid_D4_Ensemble",
            model_stage1=MODEL_STAGE1,
            device=ENV_INFO["device"],
            output_dir=OUTPUT_DIR / "val_to_test",
        )
        _callee_mutated_split = (
            [p.name for p in val_files] != _val_names_before
            or [p.name for p in test_files] != _test_names_before
        )
        if _callee_mutated_split:
            record_error("exp1_val_to_test", RuntimeError("Callee da ghi de val_files/test_files"), "callee_mutated_split")
            RUN_META["split_provenance"]["callee_mutated_split"] = True
            _post_val_pids = {extract_patient_id(p.name) for p in val_files}
            _post_test_pids = {extract_patient_id(p.name) for p in test_files}
            if _post_val_pids & _post_test_pids:
                record_error("exp1_val_to_test", RuntimeError("Sau khi goi ham, val va test trung Patient ID"), "post_call_leakage")
                val_to_test_results = None
                EXPERIMENT_STATUS["exp1_val_to_test"] = "failed_patient_overlap"
        if val_to_test_results is not None:
            EXPERIMENT_STATUS["exp1_val_to_test"] = (
                "ok_partial_callee_mutated_split" if _callee_mutated_split else "ok"
            )
    except Exception as _exp1_err:
        record_error("exp1_val_to_test", _exp1_err, "run")
        EXPERIMENT_STATUS["exp1_val_to_test"] = "failed"
        val_to_test_results = None

if val_to_test_results is not None:
    def _metric(dct, *keys):
        for _k in keys:
            if isinstance(dct, dict) and dct.get(_k) is not None:
                return float(dct[_k])
        return None

    def _fmt_pct(value, signed=False):
        if value is None:
            return "N/A"
        return f"{value:+.2f}%" if signed else f"{value:.2f}%"

    _locked = val_to_test_results.get("locked_hyperparameter", {})
    RUN_META["hyperparameters_locked"] = dict(_locked) if isinstance(_locked, dict) else {}
    _val_m = val_to_test_results.get("val_metrics", {}) or {}
    _test_m = val_to_test_results.get("test_metrics", {}) or {}
    _gap_obj = val_to_test_results.get("generalization_gap", {})
    _gap = _gap_obj.get("mIoU_gap", None) if isinstance(_gap_obj, dict) else _gap_obj
    _val_miou = _metric(_val_m, "mIoU_4cls", "mIoU")
    _test_miou = _metric(_test_m, "mIoU_4cls", "mIoU")
    print("=" * 65)
    print("KET QUA QUY TRINH KHOA HOC VAL-TO-TEST PROTOCOL:")
    print(f" - Val mIoU:            {_fmt_pct(_val_miou)}")
    print(f" - Test mIoU:           {_fmt_pct(_test_miou)}")
    print(f" - Generalization Gap:  {_fmt_pct(_gap, signed=True)}")
    print(f" - Test FwIoU:          {_fmt_pct(_metric(_test_m, 'FwIoU'))}")
    print(f" - Test Boundary F1:    {_fmt_pct(_metric(_test_m, 'Boundary_F1'))}")
    print(f" - Locked hyperparams:  {RUN_META['hyperparameters_locked']}")
    print("=" * 65)
    RUN_META["sample_sizes"]["exp1_val_to_test"].update({
        "n_val_files_evaluated": len(val_files),
        "n_test_files_evaluated": len(test_files),
        "callee_mutated_split": bool(RUN_META["split_provenance"].get("callee_mutated_split", False)),
    })
else:
    print("[FAIL] EXP1 khong co ket qua; chi tiet trong RUN_ERRORS.")

---
## Buoc 5: Thi Nghiem 2 - Multi-Magnification Degradation & Coarse-to-Fine Pareto Frontier
Mo phong suy giam do phan giai da tang ($5\times, 10\times, 20\times, 40\times$), router do phuc tap thong minh:
$$S_{\text{route}} = 0.50 \bar{H} + 0.30 R_{\text{bound}} + 0.20 P_{\text{tumor}}$$
Doi sanh duong cong toi uu Pareto giua Adaptive Routing voi Random Routing tren cac muc ngan sach tinh toan $\beta \in [0.0, 1.0]$.

In [ ]:
import json
import math
from pathlib import Path

import numpy as np

from src.colab_pkg.evaluator import run_multimag_and_pareto_benchmark
from src.colab_pkg.visualizer import plot_coarse_to_fine_pareto_curve

pareto_test_files = test_files[: (DRY_RUN_SAMPLES if SANITY_DRY_RUN else 200)]
print(f"[EXP 2] Running Multi-Mag & Pareto Benchmark on {len(pareto_test_files)} patches...")

if not pareto_test_files:
    pareto_data = None
    _exp2_skip_reason = "blocked_by_setup" if SETUP_ERROR is not None else "skipped_no_test_files"
    EXPERIMENT_STATUS["exp2_multimag_pareto"] = _exp2_skip_reason
    record_error(
        "exp2_multimag_pareto",
        ValueError("Khong co test file nao cho EXP2; khong goi benchmark de tranh self-heal tai du lieu."),
        "select",
    )
    RUN_META["sample_sizes"]["exp2_multimag_pareto"] = {
        "num_samples": 0,
        "n_patches_scheduled": 0,
        "status": _exp2_skip_reason,
    }
else:
    try:
        pareto_data = run_multimag_and_pareto_benchmark(
            engine=WSSS_ENGINE,
            model_stage2=MODEL_STAGE2,
            test_files=pareto_test_files,
            test_mask_dir=test_mask_dir,
            model_stage1=MODEL_STAGE1,
            device=ENV_INFO["device"],
            budget_steps=5,
        )
        EXPERIMENT_STATUS["exp2_multimag_pareto"] = "ok"
    except Exception as _exp2_err:
        record_error("exp2_multimag_pareto", _exp2_err, "run")
        EXPERIMENT_STATUS["exp2_multimag_pareto"] = "failed"
        pareto_data = None


# LUU Y self-reference: run_multimag_and_pareto_benchmark gan pareto_data["pareto_curve"] = pareto_data
# (src/colab_pkg/evaluator.py:1217), tuc la khoa "pareto_curve" tro ve chinh dict cha.
# Vi vay:
#   - _serialize_for_json duoi day loai bo khoa nay (tham so skip_keys) de tranh de quy vo han;
#   - Cell 18 dung lai chinh pareto_export nay, nen pareto_results.json va summary["pareto"]
#     la MOT artifact duy nhat, khong con hai ban khac nhau.
_NONFINITE_COUNT = [0]


def _serialize_for_json(obj, visited=None, skip_keys=("pareto_curve",), depth=0):
    """Serialize de ghi JSON; NaN/Inf -> None va dem vao _NONFINITE_COUNT.

    skip_keys chi ap dung o top-level (depth == 0), khong xoa khoa cung ten o dict con.
    """
    if visited is None:
        visited = set()
    oid = id(obj)
    if oid in visited:
        return None
    if isinstance(obj, np.ndarray):
        # ndarray co ca .item() va .tolist(); .item() nem ValueError khi mang > 1 phan tu.
        return _serialize_for_json(obj.tolist(), visited, skip_keys, depth + 1)
    if isinstance(obj, np.generic):
        return _serialize_for_json(obj.item(), visited, skip_keys, depth + 1)
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            _NONFINITE_COUNT[0] += 1
            return None
        return obj
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (int, str, bool)) or obj is None:
        return obj
    if isinstance(obj, dict):
        visited.add(oid)
        res = {
            str(k): _serialize_for_json(v, visited, skip_keys, depth + 1)
            for k, v in obj.items()
            if not (depth == 0 and str(k) in skip_keys)
        }
        visited.remove(oid)
        return res
    if isinstance(obj, (list, tuple)):
        visited.add(oid)
        res = [_serialize_for_json(v, visited, skip_keys, depth + 1) for v in obj]
        visited.remove(oid)
        return res
    return str(obj)


if pareto_data is not None:
    _auc_gain = pareto_data.get("auc_gain", None)
    _auc_adapt = pareto_data.get("auc_adaptive", None)
    _auc_rand = pareto_data.get("auc_random", None)
    _n_samples = pareto_data.get("num_samples", None)
    _raw_delta = None
    if _auc_adapt is not None and _auc_rand is not None:
        _raw_delta = float(_auc_adapt) - float(_auc_rand)

    print("=" * 65)
    print("KET QUA PARETO FRONTIER (Adaptive vs Random Routing):")
    if _auc_gain is None:
        print(" - AUC gain (tra ve):    N/A")
    else:
        print(f" - AUC gain (tra ve):    {float(_auc_gain):.4f} %mIoU*Budget")
    print("[NOTE] auc_gain bi clip tai 0 trong src/models/coarse_to_fine_router.py:418, cu the max(0.0, auc_adaptive - auc_random).")
    print("[NOTE] Vi bi clip, auc_gain khong bao gio am: KHONG duoc dat dau '+' de ket luan Adaptive tot hon Random.")
    if _raw_delta is None:
        print(" - Chenh lech tho:       N/A")
    else:
        print(f" - Chenh lech tho:       {_raw_delta:+.4f} (auc_adaptive - auc_random, am nghia la Adaptive kem hon Random)")
    print(f" - Sample size:          num_samples = {_n_samples if _n_samples is not None else 'N/A'}")
    print("=" * 65)

    try:
        fig_pareto, pareto_img_path = plot_coarse_to_fine_pareto_curve(
            pareto_data=pareto_data,
            output_dir=OUTPUT_DIR / "pareto",
        )
        print(f"[OK] Pareto chart exported: {pareto_img_path}")
    except Exception as _exp2_plot_err:
        record_error("exp2_multimag_pareto", _exp2_plot_err, "plot")

    try:
        pareto_export = _serialize_for_json(pareto_data)
        pareto_json_path = OUTPUT_DIR / "pareto" / "pareto_results.json"
        _pareto_text = json.dumps(pareto_export, indent=2, allow_nan=False)
        write_json_atomic(pareto_json_path, _pareto_text)
        print(f"[OK] Exported Pareto results to JSON: {pareto_json_path}")
    except (TypeError, ValueError, OSError) as _exp2_json_err:
        record_error("exp2_multimag_pareto", _exp2_json_err, "export_json")
        EXPERIMENT_STATUS["exp2_multimag_pareto"] = "failed_export"
        pareto_export = None

    RUN_META["sample_sizes"]["exp2_multimag_pareto"] = {
        "num_samples": _n_samples,
        "n_budget_steps": len(pareto_data.get("budgets", []) or []),
        "n_patches_scheduled": len(pareto_test_files),
        "auc_gain_clipped_at_zero": True,
        "auc_raw_delta": _raw_delta,
        "n_nonfinite_values": _NONFINITE_COUNT[0],
    }
else:
    print("[FAIL] EXP2 khong co ket qua; chi tiet trong RUN_ERRORS.")

---
## Buoc 6: Thi Nghiem 3 - Patient-Level Clinical Biomarker Correlation (VTP & TSR)
Luu tru va gom cum cac vung phan doan theo Patient ID:
- **VTP (Volume of Tumor Proportion)**: Ty le phan tram the tich khoi u xam lan.
- **TSR (Tumor-to-Stroma Ratio)**: Ty le U tren Mo dem - chi so tien luong doc lap chuan CAP/ASCO.
- Tinh toan he so tuong quan Pearson $r$, Spearman $\rho$, hoi quy Bland-Altman va do phu hop phan tang nguy co CAP.

In [ ]:
from collections import defaultdict
from PIL import Image
import json
import numpy as np
from src.colab_pkg.downloader import extract_patient_id
from src.colab_pkg.evaluator import run_biomarker_correlation_study, sanitize_for_json
from src.colab_pkg.visualizer import plot_biomarker_bland_altman_and_regression


def _fmt_metric(corr_dict, key, spec="{:.4f}"):
    value = corr_dict.get(key, None) if isinstance(corr_dict, dict) else None
    if value is None:
        return "N/A"
    try:
        return spec.format(float(value))
    except (TypeError, ValueError):
        return "N/A"


def _fmt_metric_n(corr_dict, key, min_n, spec="{:.4f}"):
    # src/losses_metrics/metrics.py:399-427 tra ve 0.0 nhu sentinel khi khong du mau.
    # In "0.0000" se bi doc nham la so do that, nen tra N/A khi n < min_n hoac thieu khoa.
    if not isinstance(corr_dict, dict):
        return "N/A"
    try:
        n_found = int(corr_dict.get("patient_count", 0) or 0)
    except (TypeError, ValueError):
        n_found = 0
    if n_found < min_n:
        return f"N/A (n={n_found}<{min_n})"
    return _fmt_metric(corr_dict, key, spec)


patient_preds = defaultdict(list)
patient_gts = defaultdict(list)
n_patches_used = 0
n_patches_skipped = 0
skipped_reasons = defaultdict(int)

if test_files:
    all_source_patches = sorted(test_files)
    bio_sample_source = "exp1_test_files"
else:
    all_source_patches = sorted(test_img_dir.glob("*.png")) if (test_img_dir and test_img_dir.is_dir()) else []
    bio_sample_source = "test_img_dir_fallback"
unique_test_pids = sorted(set(extract_patient_id(p.name) for p in all_source_patches))
target_bio_count = max(69, DRY_RUN_SAMPLES) if SANITY_DRY_RUN else 200
quota_bio = max(3, target_bio_count // max(1, len(unique_test_pids)))
bio_files = []
for pid in unique_test_pids:
    pid_patches = [f for f in all_source_patches if extract_patient_id(f.name) == pid]
    bio_files.extend(pid_patches[:quota_bio])
bio_files = bio_files[:target_bio_count]

print(f"[EXP 3] Aggregating Patient Biomarkers across {len(bio_files)} candidate patches ({len(unique_test_pids)} distinct patients)...")

if len(bio_files) == 0:
    record_error("exp3_biomarker", RuntimeError("Khong co patch nao kha dung cho tuong quan biomarker."), "select")
    EXPERIMENT_STATUS["exp3_biomarker"] = "skipped_no_patches"
else:
    for p in bio_files:
        pid = extract_patient_id(p.name)
        mask_path = test_mask_dir / p.name
        if not mask_path.is_file():
            n_patches_skipped += 1
            skipped_reasons["mask_missing"] += 1
            continue
        try:
            gt_arr = np.array(Image.open(mask_path))
            rgb_np = np.array(Image.open(p).convert("RGB"))
            pred_arr = WSSS_ENGINE.predict_m6(MODEL_STAGE2, None, raw_rgb=rgb_np, model_stage1=MODEL_STAGE1)
        except (OSError, ValueError, RuntimeError) as _bio_patch_err:
            n_patches_skipped += 1
            skipped_reasons["read_or_inference_error"] += 1
            record_error("exp3_biomarker", _bio_patch_err, f"patch {p.name}")
            continue
        patient_preds[pid].append(pred_arr)
        patient_gts[pid].append(gt_arr)
        n_patches_used += 1

    try:
        if len(patient_gts) == 0:
            raise RuntimeError("Khong co benh nhan nao co du ca prediction va mask; khong the tinh tuong quan.")
        bio_results = run_biomarker_correlation_study(
            patient_predictions=patient_preds,
            patient_ground_truths=patient_gts,
            output_csv_path=OUTPUT_DIR / "biomarkers" / "patient_biomarker_correlation.csv",
        )
        fig_bio, bio_img_path = plot_biomarker_bland_altman_and_regression(
            pred_biomarkers=bio_results["pred_biomarkers"],
            gt_biomarkers=bio_results["gt_biomarkers"],
            output_dir=OUTPUT_DIR / "biomarkers",
        )
        corr = bio_results["correlation_metrics"]
        corr["n_patches_used"] = int(n_patches_used)
        corr["n_patches_skipped"] = int(n_patches_skipped)
        corr["n_patients"] = int(len(patient_gts))
        corr["patches_scheduled"] = int(len(bio_files))
        corr["skipped_reasons"] = dict(skipped_reasons)
        corr["bio_sample_source"] = bio_sample_source
        corr["n_patches_overlap_with_exp1"] = len({p.name for p in all_source_patches} & {p.name for p in test_files})
        corr["sample_size_note"] = (
            "n_patches_used la so patch thuc su dua vao tuong quan; n_patches_skipped bi bo "
            "vi thieu mask hoac loi doc/suy luan."
        )
        bio_json_path = OUTPUT_DIR / "biomarkers" / "biomarker_correlation.json"
        bio_json_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            # json.dumps truoc roi moi write_text: khong de lai file 0 byte khi serialize loi.
            _bio_text = json.dumps(sanitize_for_json(corr), indent=2, allow_nan=False)
            bio_json_path.write_text(_bio_text, encoding="utf-8")
            print(f"[OK] Exported Biomarker Correlation results: {bio_json_path}")
        except (TypeError, ValueError) as _bio_json_err:
            record_error("exp3_biomarker", _bio_json_err, "export_json")
            EXPERIMENT_STATUS["exp3_biomarker"] = "failed_export"
        else:
            EXPERIMENT_STATUS["exp3_biomarker"] = "ok"
        RUN_META["sample_sizes"]["exp3_biomarker"] = {
            "n_patches_used": int(n_patches_used),
            "n_patches_skipped": int(n_patches_skipped),
            "n_patients": int(len(patient_gts)),
            "skipped_reasons": dict(skipped_reasons),
        }
    except Exception as _exp3_err:
        record_error("exp3_biomarker", _exp3_err, "run")
        EXPERIMENT_STATUS["exp3_biomarker"] = "failed"
        bio_results = None

if bio_results is not None:
    corr = bio_results["correlation_metrics"]
    _n_pat = corr.get("patient_count", "N/A")
    _n_tum = corr.get("tumor_positive_patient_count", "N/A")
    print("=" * 65)
    print(f"KET QUA TUONG QUAN CHI SO SINH HOC LAM SANG ({_n_pat} Patients, {_n_tum} Tumor+):")
    print(f" - Sample size:           {n_patches_used} patches used | {n_patches_skipped} skipped | {len(patient_gts)} patients")
    if skipped_reasons:
        print(f" - Ly do bo qua:          {dict(skipped_reasons)}")
    print(f" - VTP Pearson r:         {_fmt_metric_n(corr, 'vtp_pearson_r', 2)} | ICC(2,1): {_fmt_metric_n(corr, 'vtp_icc', 3)} | CCC: {_fmt_metric_n(corr, 'vtp_ccc', 3)}")
    print(f" - TSR Pearson r:         {_fmt_metric_n(corr, 'tsr_pearson_r', 2)} | ICC(2,1): {_fmt_metric_n(corr, 'tsr_icc', 3)} | CCC: {_fmt_metric_n(corr, 'tsr_ccc', 3)}")
    print(f" - VTP Bias / 95% LoA:    {_fmt_metric_n(corr, 'vtp_bias', 3, '{:+.2f}%')} [{_fmt_metric_n(corr, 'vtp_loa_lower', 3, '{:+.2f}%')}, {_fmt_metric_n(corr, 'vtp_loa_upper', 3, '{:+.2f}%')}]")
    print(f" - CAP Risk Concordance:  {_fmt_metric_n(corr, 'cap_risk_concordance_pct', 3, '{:.2f}%')}")
    print("=" * 65)
else:
    print("[FAIL] EXP3 khong co ket qua; chi tiet trong RUN_ERRORS.")

---
## Buoc 7: Thi Nghiem 4 - In-Domain LUAD + BCSS Transfer Baseline (Ung Thu Phoi)
Thi nghiem nay gom HAI ket qua khac nhau ve ban chat, in duoi hai nhan rieng:
- **[IN-DOMAIN LUAD]**: checkpoint LUAD-HistoSeg duoc tai qua `ensure_luad_checkpoints` va nap qua `load_luad_models`, roi danh gia tren chinh tap LUAD test. Day KHONG phai zero-shot; day la baseline in-domain tren Lung Adenocarcinoma (4 classes: Tumor, Stroma, Normal, Necrosis).
- **[TRANSFER BCSS -> LUAD]**: dung trong so BCSS (Breast) chay truc tiep tren LUAD (Lung) sau khi remap nhan; `run_cross_dataset_luad_benchmark` tra ve ket qua nay duoi khoa `zero_shot_bcss_to_luad_transfer`. Day moi la phep do chuyen co quan.
- Khong duoc gop hai con so nay lam mot, va khong duoc goi ket qua in-domain la "zero-shot".
- Gioi han: LUAD in-domain van la co quan khac BCSS, nhung checkpoint da duoc huan luyen tren LUAD; vi vay ket qua nay do chat luong checkpoint LUAD, khong do kha nang tong quat hoa cheo co quan.

In [ ]:
from src.colab_pkg.downloader import ensure_luad_checkpoints, ensure_luad_dataset
from src.colab_pkg.evaluator import run_cross_dataset_luad_benchmark
from src.colab_pkg.model_loader import load_luad_models


def _luad_pct(value):
    if value is None:
        return "N/A"
    try:
        return f"{float(value):.2f}%"
    except (TypeError, ValueError):
        return "N/A"


def _luad_get(dct, *keys):
    if not isinstance(dct, dict):
        return None
    for _k in keys:
        if dct.get(_k) is not None:
            return dct[_k]
    return None


luad_test_files = []
try:
    luad_root, luad_img_dir, luad_mask_dir = ensure_luad_dataset(
        data_root=WORK_ROOT / "LUAD-HistoSeg",
        drive_cache_dir=None,
    )
    luad_ckpts = ensure_luad_checkpoints(
        checkpoint_dir=WORK_ROOT / "checkpoints",
        drive_cache_dir=None,
    )
    model_luad_stage2, model_luad_stage1, _ = load_luad_models(
        stage2_ckpt=luad_ckpts.get("stage2"),
        stage1_ckpt=luad_ckpts.get("stage1"),
        device=ENV_INFO["device"],
    )
    luad_test_files = sorted(luad_img_dir.glob("*.png"))
    if SANITY_DRY_RUN:
        luad_test_files = luad_test_files[:20]

    print(f"[EXP 4] In-domain LUAD benchmark tren {len(luad_test_files)} patches (checkpoint LUAD, KHONG phai zero-shot)...")
    luad_results = run_cross_dataset_luad_benchmark(
        engine=WSSS_ENGINE,
        model_luad_stage2=model_luad_stage2,
        luad_test_files=luad_test_files,
        luad_mask_dir=luad_mask_dir,
        model_luad_stage1=model_luad_stage1,
        bcss_model_stage2=MODEL_STAGE2,
        bcss_model_stage1=MODEL_STAGE1,
        device=ENV_INFO["device"],
        output_dir=OUTPUT_DIR / "luad",
    )
    EXPERIMENT_STATUS["exp4_luad_in_domain"] = "ok"
except Exception as _exp4_err:
    record_error("exp4_luad_in_domain", _exp4_err, "run")
    EXPERIMENT_STATUS["exp4_luad_in_domain"] = "failed"
    EXPERIMENT_STATUS["exp4b_bcss_to_luad_transfer"] = "blocked_by_exp4"
    RUN_META["sample_sizes"]["exp4b_bcss_to_luad_transfer"] = {
        "n_luad_test_files_scheduled": 0,
        "blocked_by": "exp4_luad_in_domain",
    }
    luad_results = None
    print("[FAIL] EXP4 in-domain LUAD khong chay duoc; loi da ghi vao RUN_ERRORS, khong bi nuot.")

if luad_results is not None:
    _l_miou = _luad_get(luad_results, "mIoU_4cls", "mIoU")
    _l_dice = _luad_get(luad_results, "mDice_4cls", "mDice")
    _l_fwiou = _luad_get(luad_results, "FwIoU")
    print("=" * 65)
    print("[IN-DOMAIN LUAD] Ket qua baseline tren LUAD-HistoSeg (checkpoint LUAD):")
    print(f" - mIoU:   {_luad_pct(_l_miou)}")
    print(f" - Dice:   {_luad_pct(_l_dice)}")
    print(f" - FwIoU:  {_luad_pct(_l_fwiou)}")

    luad_transfer_results = luad_results.get("zero_shot_bcss_to_luad_transfer", None)
    if luad_transfer_results is not None:
        EXPERIMENT_STATUS["exp4b_bcss_to_luad_transfer"] = "ok"
        _t_miou = _luad_get(luad_transfer_results, "mIoU_4cls", "mIoU")
        _t_dice = _luad_get(luad_transfer_results, "mDice_4cls", "mDice")
        print("[TRANSFER BCSS -> LUAD] Ket qua chuyen trong so BCSS sang LUAD (remap nhan):")
        print(f" - mIoU:   {_luad_pct(_t_miou)}")
        print(f" - Dice:   {_luad_pct(_t_dice)}")
    else:
        EXPERIMENT_STATUS["exp4b_bcss_to_luad_transfer"] = "missing_key"
        record_error(
            "exp4b_bcss_to_luad_transfer",
            KeyError("zero_shot_bcss_to_luad_transfer khong co trong ket qua tra ve"),
            "read_result",
        )
    print("=" * 65)

    RUN_META["sample_sizes"]["exp4_luad_in_domain"] = {
        "n_luad_test_files_scheduled": len(luad_test_files),
        "evaluation_protocol": luad_results.get("evaluation_protocol", None),
    }
    RUN_META["sample_sizes"]["exp4b_bcss_to_luad_transfer"] = {
        "n_luad_test_files_scheduled": len(luad_test_files),
        "source": "zero_shot_bcss_to_luad_transfer",
    }
else:
    print("[FAIL] EXP4 khong co ket qua; chi tiet trong RUN_ERRORS.")

---
## Buoc 8: Thi Nghiem 5 - Whole-ROI Evaluation (coverage reported)
Kiem dinh tren cac tieu ban Whole-ROI co Ground-Truth chu thich day du tu bac si giai phau benh (BCSS Annotated Cohort):
- Canvas duoc lap day bang du doan coarse tu quet Low-Mag (Coarse Base Map); chi cac candidate patch duoc suy luan o 40x moi duoc ghi de len canvas.
- Vi vay do phu that KHONG phai 100%: moi slide ghi lai `patches_inferred`, `total_possible_patches`, `selected_candidate_patches`, `coverage_ratio`, `is_partial_coverage`.
- Khi `coverage_ratio < 0.999` thi `metric_valid = False` va ket qua duoc in duoi nhan `[PARTIAL]`, khong duoc trich dan nhu ket qua Whole-ROI day du.
- So slide: SANITY dung 3 Whole-ROI; FULL dung min(10, so slide co trong cohort).
- Tinh toan mIoU, Dice, Pixel Accuracy, FwIoU va Boundary F1 tren phan ROI da duoc suy luan that, kem canh bao coverage.

In [ ]:
import json
import numpy as np
from PIL import Image
from src.colab_pkg.downloader import download_bcss_whole_roi_cohort
from src.colab_pkg.evaluator import sanitize_for_json
from src.colab_pkg.visualizer import plot_whole_roi_ground_truth_comparison
from src.pipeline.wsi_lowmag_highmag_pipeline import WSILowMagHighMagPipeline

pipeline_wsi = WSILowMagHighMagPipeline(
    patch_size=224,
    lowmag_target_size=(1024, 1024),
    tissue_threshold=0.10,
    engine=WSSS_ENGINE,
    device=ENV_INFO["device"],
)

COVERAGE_TOLERANCE = 0.999


def _coverage_from_report(rep, max_patches_cap):
    """Doc GRID coverage THAT tu WSIProcessingReport (khong doan).

    selected_candidate_patches cua rep la so candidate SAU khi bi cat boi max_patches_to_infer
    (src/pipeline/wsi_lowmag_highmag_pipeline.py:480-481, 613), nen khong the dung no de phat hien
    viec cat bot. coverage_ratio duoc tinh tren total_possible_patches, tuc toan bo luoi patch
    ly thuyet cua ROI; phan con lai la do canvas lap bang du doan coarse Stage-1.
    LUU Y: coverage_ratio la GRID coverage. metric_valid duoc quyet dinh boi
    tissue_coverage_ratio (xem _tissue_coverage_from_report) va cap_truncated; KHONG dung
    candidate_coverage_ratio vi no luon ~1.0 sau khi bi cat.
    """
    patches_inferred = len(getattr(rep, "patch_stats", []) or [])
    total_possible = int(getattr(rep, "total_possible_patches", 0) or 0)
    selected_candidates = int(getattr(rep, "selected_candidate_patches", 0) or 0)
    coverage = (patches_inferred / float(total_possible)) if total_possible > 0 else 0.0
    candidate_coverage = (patches_inferred / float(selected_candidates)) if selected_candidates > 0 else 0.0
    return {
        "patches_inferred": patches_inferred,
        "total_possible_patches": total_possible,
        "selected_candidate_patches": selected_candidates,
        "coverage_ratio": round(coverage, 6),
        "candidate_coverage_ratio": round(candidate_coverage, 6),
        "max_patches_to_infer": max_patches_cap,
        "compute_reduction_pct": float(getattr(rep, "compute_reduction_pct", 0.0) or 0.0),
    }


def _tissue_coverage_from_report(rep, gt_std, patch_size=224):
    """Do phan tram pixel GT tissue nam trong cac patch da suy luan that.

    patch_stats chi co x, y (toa do anh goc), khong co width/height, nen dung
    patch_size cua pipeline. macro_dimensions / original_dimensions cho ty le
    tu anh goc sang canvas macro.
    """
    orig = getattr(rep, "original_dimensions", None)
    macro = getattr(rep, "macro_dimensions", None)
    stats = list(getattr(rep, "patch_stats", []) or [])
    if not orig or not macro or not stats:
        return {"tissue_coverage_ratio": 0.0, "covered_tissue_px": 0,
                "total_tissue_px": 0, "coverage_basis": "unavailable",
                "coverage_resolution": None}
    # read_slide_overview tra (orig_width, orig_height); lowmag_target_size la (W, H) theo PIL.
    orig_w, orig_h = int(orig[0]), int(orig[1])
    macro_w, macro_h = int(macro[0]), int(macro[1])
    if orig_w <= 0 or orig_h <= 0 or macro_w <= 0 or macro_h <= 0:
        return {"tissue_coverage_ratio": 0.0, "covered_tissue_px": 0,
                "total_tissue_px": 0, "coverage_basis": "invalid_dimensions",
                "coverage_resolution": None}
    gt = np.asarray(gt_std)
    gt_macro = np.array(
        Image.fromarray(gt.astype(np.uint8)).resize((macro_w, macro_h), Image.Resampling.NEAREST)
    )
    tissue = (gt_macro >= 0) & (gt_macro < 4)
    total = int(tissue.sum())
    if total == 0:
        return {"tissue_coverage_ratio": 0.0, "covered_tissue_px": 0,
                "total_tissue_px": 0, "coverage_basis": "no_gt_tissue",
                "coverage_resolution": [macro_w, macro_h]}
    cov = np.zeros((macro_h, macro_w), dtype=bool)
    sx = macro_w / float(orig_w)
    sy = macro_h / float(orig_h)
    for st in stats:
        px = int(st["x"])
        py = int(st["y"])
        pw = min(patch_size, max(1, orig_w - px))
        ph = min(patch_size, max(1, orig_h - py))
        x0 = max(0, int(px * sx))
        x1 = min(macro_w, int((px + pw) * sx))
        y0 = max(0, int(py * sy))
        y1 = min(macro_h, int((py + ph) * sy))
        if x1 > x0 and y1 > y0:
            cov[y0:y1, x0:x1] = True
    covered = int((tissue & cov).sum())
    return {"tissue_coverage_ratio": round(covered / float(total), 6),
            "covered_tissue_px": covered, "total_tissue_px": total,
            "coverage_basis": "gt_tissue_pixels_inside_inferred_patches",
            "coverage_resolution": [macro_w, macro_h]}


try:
    if SETUP_ERROR is not None:
        raise RuntimeError("Setup that bai; khong tai BCSS Whole-ROI.")
    roi_cohort = download_bcss_whole_roi_cohort(dest_dir=WORK_ROOT / "bcss_whole_roi")
except Exception as _roi_dl_err:
    record_error("exp5_whole_roi", _roi_dl_err, "download")
    EXPERIMENT_STATUS["exp5_whole_roi"] = "blocked_by_setup" if SETUP_ERROR is not None else "failed"
    roi_cohort = []

per_slide_reports = []
coverage_records = []
agg_cm4 = np.zeros((4, 4), dtype=np.int64)
roi_metrics = None
max_cap = 64 if SANITY_DRY_RUN else None
eval_rois = roi_cohort[: (3 if SANITY_DRY_RUN else min(10, len(roi_cohort)))] if roi_cohort else []

if not eval_rois:
    if EXPERIMENT_STATUS["exp5_whole_roi"] == "not_run":
        record_error("exp5_whole_roi", FileNotFoundError("Khong tim thay mau Whole-ROI trong thu muc."), "cohort")
        EXPERIMENT_STATUS["exp5_whole_roi"] = "skipped_no_roi"
else:
    for idx_roi, sample_roi in enumerate(eval_rois):
        slide_id = sample_roi.get("patient_id", f"WHOLE_ROI_{idx_roi + 1}")
        try:
            roi_img = np.array(Image.open(sample_roi["img"]).convert("RGB"))
            roi_gt_raw = np.array(Image.open(sample_roi["mask"]))
            stitched_pred, rep = pipeline_wsi.run_end_to_end_wsi(
                slide_input=roi_img,
                model_stage2=MODEL_STAGE2,
                model_stage1=MODEL_STAGE1,
                slide_id=slide_id,
                max_patches_to_infer=max_cap,
            )
            roi_gt_std = pipeline_wsi.standardize_bcss_roi_mask(
                roi_gt_raw, pred_mask=stitched_pred, label_convention="bcss_raw"
            )
            cov = _coverage_from_report(rep, max_cap)
            cov["grid_coverage_ratio"] = cov["coverage_ratio"]
            cov.update(_tissue_coverage_from_report(rep, roi_gt_std))
            cov["cap_truncated"] = bool(max_cap is not None and cov["patches_inferred"] >= max_cap)
            cov["cap_semantics"] = "conservative_upper_bound: patches_inferred >= max_patches_to_infer"
            cov["is_partial_coverage"] = bool(
                cov["tissue_coverage_ratio"] < COVERAGE_TOLERANCE or cov["cap_truncated"]
            )
            slide_m = pipeline_wsi.evaluate_wsi_with_ground_truth_roi(
                stitched_pred, roi_gt_std, label_convention="wsss_4cls"
            )
        except Exception as _roi_err:
            record_error("exp5_whole_roi", _roi_err, f"slide {slide_id}")
            continue

        slide_m["slide_id"] = slide_id
        _slide_support = [
            int(v) for v in np.array(slide_m["confusion_matrix_4cls"], dtype=np.int64).sum(axis=1)
        ]
        slide_m["gt_support_4cls"] = _slide_support
        slide_m["classes_absent_in_gt"] = [i for i, v in enumerate(_slide_support) if v == 0]
        slide_m["coverage"] = cov
        slide_m["is_partial_coverage"] = cov["is_partial_coverage"]
        slide_m["metric_valid"] = bool(
            not cov["is_partial_coverage"] and sum(_slide_support) > 0
        )
        per_slide_reports.append(slide_m)
        coverage_records.append(cov)
        agg_cm4 += np.array(slide_m["confusion_matrix_4cls"], dtype=np.int64)
        _tag = "[PARTIAL]" if cov["is_partial_coverage"] else "[OK]"
        print(
            f"{_tag} {slide_id}: tissue coverage {cov['tissue_coverage_ratio'] * 100.0:.2f}% | "
            f"grid {cov['coverage_ratio'] * 100.0:.2f}% "
            f"({cov['patches_inferred']}/{cov['total_possible_patches']} patches, cap={max_cap}) | "
            f"mIoU {float(slide_m['mIoU_4cls']):.2f}%"
        )
        if idx_roi == 0:
            # Truyen GT RAW: ham tu goi standardize_bcss_roi_mask(label_convention="auto").
            # Neu truyen roi_gt_std (da o thang 0..4) thi heuristic auto remap lan hai.
            try:
                fig_roi, roi_fig_path = plot_whole_roi_ground_truth_comparison(
                    rgb_image=roi_img,
                    pred_mask=stitched_pred,
                    gt_mask=roi_gt_raw,
                    output_dir=OUTPUT_DIR / "whole_roi",
                )
            except Exception as _roi_plot_err:
                record_error("exp5_whole_roi", _roi_plot_err, f"plot {slide_id}")

if per_slide_reports:
    roi_metrics = dict(per_slide_reports[0])
    # per-class cua slide 0 khong dai dien cohort: xoa va thay bang per-class tren agg_cm4.
    roi_metrics.pop("per_class_iou", None)
    roi_metrics.pop("per_class_dice", None)
    roi_metrics["evaluated_slides_count"] = len(per_slide_reports)
    roi_metrics["mIoU_4cls"] = float(np.mean([r["mIoU_4cls"] for r in per_slide_reports]))
    roi_metrics["mIoU_4cls_std"] = float(np.std([r["mIoU_4cls"] for r in per_slide_reports]))
    roi_metrics["mDice_4cls"] = float(np.mean([r["mDice_4cls"] for r in per_slide_reports]))
    roi_metrics["Pixel_Accuracy"] = float(np.mean([r["Pixel_Accuracy"] for r in per_slide_reports]))
    roi_metrics["FwIoU"] = float(np.mean([r["FwIoU"] for r in per_slide_reports]))
    roi_metrics["Boundary_F1"] = float(np.mean([r["Boundary_F1"] for r in per_slide_reports]))
    roi_metrics["confusion_matrix_4cls"] = agg_cm4.tolist()
    _tp = np.diag(agg_cm4).astype(np.float64)
    _den_iou = agg_cm4.sum(axis=0) + agg_cm4.sum(axis=1) - _tp
    _den_dice = 2.0 * _tp + (agg_cm4.sum(axis=0) - _tp) + (agg_cm4.sum(axis=1) - _tp)
    per_class_iou_cohort = [float(_tp[i] / _den_iou[i] * 100.0) if _den_iou[i] > 0 else None for i in range(4)]
    per_class_dice_cohort = [float(2.0 * _tp[i] / _den_dice[i] * 100.0) if _den_dice[i] > 0 else None for i in range(4)]
    gt_support_4cls = [int(v) for v in agg_cm4.sum(axis=1)]
    roi_metrics["per_class_iou_cohort"] = per_class_iou_cohort
    roi_metrics["per_class_dice_cohort"] = per_class_dice_cohort
    roi_metrics["gt_support_4cls"] = gt_support_4cls
    roi_metrics["classes_absent_in_gt"] = [i for i, v in enumerate(gt_support_4cls) if v == 0]
    roi_metrics["slide_id"] = "AGGREGATE"
    roi_metrics["per_slide_reports"] = per_slide_reports
    roi_metrics["mIoU_4cls_std_note"] = "std la do lech giua cac slide, KHONG phai do lech tai lap (single seed)."
    roi_metrics["coverage"] = {
        "per_slide": coverage_records,
        "min_coverage_ratio": float(min(c["tissue_coverage_ratio"] for c in coverage_records)),
        "mean_coverage_ratio": float(np.mean([c["tissue_coverage_ratio"] for c in coverage_records])),
        "max_coverage_ratio": float(max(c["tissue_coverage_ratio"] for c in coverage_records)),
        "min_grid_coverage_ratio": float(min(c["grid_coverage_ratio"] for c in coverage_records)),
        "mean_grid_coverage_ratio": float(np.mean([c["grid_coverage_ratio"] for c in coverage_records])),
        "max_grid_coverage_ratio": float(max(c["grid_coverage_ratio"] for c in coverage_records)),
        "any_cap_truncated": bool(any(c["cap_truncated"] for c in coverage_records)),
        "coverage_tolerance": COVERAGE_TOLERANCE,
        "any_partial_coverage": bool(any(c["is_partial_coverage"] for c in coverage_records)),
        "max_patches_to_infer": max_cap,
        "coverage_basis": (
            "min/mean/max_coverage_ratio = tissue_coverage_ratio (GT tissue pixels inside "
            "inferred patches); *_grid_coverage_ratio = inferred patches / total theoretical grid"
        ),
    }
    roi_metrics["is_partial_coverage"] = roi_metrics["coverage"]["any_partial_coverage"]
    roi_metrics["metric_valid"] = bool(
        not roi_metrics["is_partial_coverage"] and sum(gt_support_4cls) > 0
    )
    _validity_notes = []
    if roi_metrics["is_partial_coverage"]:
        _validity_notes.append(
            "tissue_coverage_ratio < 0.999 hoac cap_truncated: canvas duoc lap bang du doan coarse "
            "Stage-1 o phan khong suy luan 40x; mIoU/Dice/FwIoU khong duoc coi la ket qua Whole-ROI day du."
        )
    if sum(gt_support_4cls) == 0:
        _validity_notes.append(
            "GT Whole-ROI khong co pixel tissue (gt_support_4cls toan 0): metric khong hop le."
        )
    elif roi_metrics["classes_absent_in_gt"]:
        _validity_notes.append(
            f"Cac lop {roi_metrics['classes_absent_in_gt']} khong co pixel GT; "
            "per-class IoU tuong ung la None, khong phai 0.0."
        )
    if _validity_notes:
        roi_metrics["metric_validity_note"] = " ".join(_validity_notes)

    _exp5_had_errors = any(e.get("experiment") == "exp5_whole_roi" for e in RUN_ERRORS)
    if not roi_metrics["metric_valid"] or _exp5_had_errors:
        EXPERIMENT_STATUS["exp5_whole_roi"] = "ok_partial_coverage"
    else:
        EXPERIMENT_STATUS["exp5_whole_roi"] = "ok"

    RUN_META["sample_sizes"]["exp5_whole_roi"] = {
        "n_slides": len(per_slide_reports),
        "n_slides_scheduled": len(eval_rois),
        "min_coverage_ratio": roi_metrics["coverage"]["min_coverage_ratio"],
        "mean_coverage_ratio": roi_metrics["coverage"]["mean_coverage_ratio"],
        "min_tissue_coverage_ratio": roi_metrics["coverage"]["min_coverage_ratio"],
        "mean_tissue_coverage_ratio": roi_metrics["coverage"]["mean_coverage_ratio"],
        "min_grid_coverage_ratio": roi_metrics["coverage"]["min_grid_coverage_ratio"],
        "any_cap_truncated": roi_metrics["coverage"]["any_cap_truncated"],
        "max_patches_to_infer": max_cap,
        "metric_valid": roi_metrics["metric_valid"],
    }

    _roi_nonfinite = []
    _roi_payload = nan_to_none(roi_metrics, found=_roi_nonfinite)
    _roi_payload["nonfinite_fields"] = _roi_nonfinite
    if _roi_nonfinite:
        record_error(
            "exp5_whole_roi",
            ValueError(f"{len(_roi_nonfinite)} truong NaN/Inf trong Whole-ROI metrics"),
            "nan_to_none",
        )
        EXPERIMENT_STATUS["exp5_whole_roi"] = "ok_partial_coverage"
    roi_metrics = _roi_payload
    whole_roi_json = OUTPUT_DIR / "whole_roi" / "whole_roi_metrics.json"
    try:
        _roi_text = json.dumps(sanitize_for_json(roi_metrics), indent=2, allow_nan=False)
        write_json_atomic(whole_roi_json, _roi_text)
        print(f"[OK] Exported Whole-ROI metrics ({len(per_slide_reports)} slides): {whole_roi_json}")
    except (TypeError, ValueError, OSError) as _roi_json_err:
        record_error("exp5_whole_roi", _roi_json_err, "export_json")
        EXPERIMENT_STATUS["exp5_whole_roi"] = "failed_export"

    _tag2 = "[PARTIAL]" if not roi_metrics["metric_valid"] else "[OK]"
    print(f"{_tag2} Whole-ROI Cohort mIoU: {roi_metrics['mIoU_4cls']:.2f}% (+/- {roi_metrics['mIoU_4cls_std']:.2f}% across slides) | FwIoU: {roi_metrics['FwIoU']:.2f}%")
    print(f"{_tag2} Tissue coverage: min {roi_metrics['coverage']['min_coverage_ratio'] * 100.0:.2f}% | mean {roi_metrics['coverage']['mean_coverage_ratio'] * 100.0:.2f}% | grid mean {roi_metrics['coverage']['mean_grid_coverage_ratio'] * 100.0:.2f}% | metric_valid={roi_metrics['metric_valid']}")
elif EXPERIMENT_STATUS["exp5_whole_roi"] not in ("skipped_no_roi", "blocked_by_setup"):
    EXPERIMENT_STATUS["exp5_whole_roi"] = "failed"
    print("[FAIL] Khong co slide Whole-ROI nao tao duoc metric; chi tiet trong RUN_ERRORS.")

---
## Buoc 9: Dong Goi Ket Qua & Tu Dong Tai Truc Tiep Ve May Tinh (Direct Browser Download)
Toan bo ket qua nghien cuu khoa hoc (cac file JSON chi so, bang tong hop CSV, bieu do xuat ban 300-DPI chat luong cao PNG) se duoc tu dong nen thanh goi ZIP va kich hoat trinh duyet tai truc tiep ve may tinh cua ban ma hoan toan khong can Google Drive.

In [ ]:
import json
import math
import os
import sys
import zipfile
from pathlib import Path

import numpy as np

from src.colab_pkg.evaluator import sanitize_for_json

PACKAGE_VERSIONS = {
    "python": sys.version.split()[0],
    "torch": str(RUN_META.get("torch_version", "unknown")),
    "numpy": str(getattr(np, "__version__", "unknown")),
}


def _pct(value, signed=False):
    if value is None:
        return "N/A"
    try:
        return f"{float(value):+.2f}%" if signed else f"{float(value):.2f}%"
    except (TypeError, ValueError):
        return "N/A"


def _num(value, spec="{:.4f}"):
    if value is None:
        return "N/A"
    try:
        return spec.format(float(value))
    except (TypeError, ValueError):
        return "N/A"


def _metric_from(dct, *keys):
    if not isinstance(dct, dict):
        return None
    for _k in keys:
        if dct.get(_k) is not None:
            return dct[_k]
    return None


def _reject_constant(token):
    raise ValueError(f"JSON chua hang so khong huu han: {token}")


limitations = [
    "Single seed (seed=42): khong co mu +/- sigma va khong do do lech tai lap giua cac seed.",
    f"Run mode = {RUN_MODE}: "
    + (
        "SANITY la ban chay mau nhanh, khong dung de bao cao ket qua cuoi cung."
        if SANITY_DRY_RUN
        else "FULL la ban chay day du theo cau hinh hien tai."
    ),
    "Whole-ROI: khi tissue_coverage_ratio < 0.999 hoac cap_truncated thi metric_valid=False; grid_coverage_ratio chi de tham khao, khong dung de quyet dinh metric_valid.",
    "EXP4 la in-domain LUAD (checkpoint LUAD); ket qua BCSS -> LUAD transfer duoc bao cao rieng, KHONG phai zero-shot in-domain.",
    "mIoU_4cls_std cua Whole-ROI la do lech giua cac slide, khong phai do lech tai lap.",
    "Pareto auc_gain bi clip tai 0 trong src/models/coarse_to_fine_router.py:418; chenh lech tho duoc ghi trong sample_sizes.",
    "Bieu do Pareto co annotation 'Sweet Spot' co dinh trong src/colab_pkg/visualizer.py:536-549, khong phan anh auc_gain.",
    "run_val_to_test_scientific_protocol co the ghi de val_files/test_files khi self-heal; notebook so sanh truoc/sau va ghi co callee_mutated_split.",
    "Whole-ROI co the thieu lop trong GT; gt_support_4cls ghi so pixel GT tung lop.",
    "OUTPUT_DIR duoc dat rieng theo RUN_MODE va RUN_TIMESTAMP, nen JSON cua lan chay truoc khong lot vao validation/ZIP.",
    "Cac JSON do evaluator tu ghi (val_to_test, LUAD) dung sanitize_for_json cua src; notebook chi kiem tra tinh hop le JSON, khong the phan biet 0.0 that va 0.0 thay the cho NaN.",
    "cap_truncated la canh bao bao thu: patches_inferred >= max_patches_to_infer, ke ca khi so candidate dung bang cap.",
]

provenance = {
    "git_commit": RUN_META.get("git_commit"),
    "seed": RUN_META.get("seed"),
    "run_mode": RUN_META.get("run_mode"),
    "sanity_dry_run": RUN_META.get("sanity_dry_run"),
    "run_started_utc": RUN_META.get("run_started_utc"),
    "notebook_path": RUN_META.get("notebook_path"),
    "split_provenance": RUN_META.get("split_provenance"),
    "sample_sizes": RUN_META.get("sample_sizes"),
    "hyperparameters_locked": RUN_META.get("hyperparameters_locked"),
    "packages": PACKAGE_VERSIONS,
    "device": RUN_META.get("device"),
    "gpu_name": RUN_META.get("gpu_name"),
    "gpu_vram_gb": RUN_META.get("gpu_vram_gb"),
    "git_sync_status": RUN_META.get("git_sync_status"),
    "run_warnings": list(RUN_WARNINGS),
    "work_root": RUN_META.get("work_root"),
}

# 1) Validate toan bo JSON da sinh boi Cell 10/12/14/16 TRUOC khi xay summary.
_invalid_jsons = []
_json_files = sorted(OUTPUT_DIR.rglob("*.json"))
for _jp in _json_files:
    try:
        json.loads(_jp.read_text(encoding="utf-8"), parse_constant=_reject_constant)
    except (ValueError, UnicodeDecodeError, OSError) as _json_err:
        _invalid_jsons.append(str(_jp))
        record_error("export_validate", _json_err, str(_jp))
print(f"[CHECK] Da validate {len(_json_files)} file JSON truoc khi xay summary.")
if _invalid_jsons:
    print("[FAIL] JSON khong hop le, KHONG dong goi ZIP: " + "; ".join(_invalid_jsons))
    EXPERIMENT_STATUS["export_summary"] = "failed_json_validation"

master_summary = {
    "project": "SE374.R11 Medical WSSS Scientific Research Runner",
    "run": RUN_META,
    "provenance": provenance,
    "device": str(ENV_INFO.get("device", "cpu")),
    "sanity_dry_run": SANITY_DRY_RUN,
    "run_mode": RUN_MODE,
    "val_to_test": val_to_test_results if val_to_test_results is not None else {},
    # pareto_export da loai bo "pareto_curve" self-reference trong Cell 10, nen summary["pareto"]
    # va pareto_results.json la cung mot artifact.
    "pareto": pareto_export if pareto_export is not None else {},
    "biomarkers": bio_results if bio_results is not None else {},
    "luad_in_domain": luad_results if luad_results is not None else {},
    "bcss_to_luad_transfer": luad_transfer_results if luad_transfer_results is not None else {},
    "whole_roi": roi_metrics if roi_metrics is not None else {},
    "limitations": limitations,
}

# 2) Lam sach NaN/Inf TRUOC khi serialize; khong de sanitize_for_json doi chung thanh 0.0.
_nonfinite = []
master_summary = nan_to_none(master_summary, found=_nonfinite)
master_summary["nonfinite_fields"] = _nonfinite
if _nonfinite:
    record_error("export_summary", ValueError(f"{len(_nonfinite)} truong NaN/Inf"), "nan_to_none")

# Ghi trang thai export TRUOC khi dong summary de summary["experiment_status"] phan anh dung.
if _invalid_jsons:
    EXPERIMENT_STATUS["export_summary"] = "failed_json_validation"
elif _nonfinite:
    EXPERIMENT_STATUS["export_summary"] = "failed_nonfinite"
else:
    EXPERIMENT_STATUS["export_summary"] = "ok"

# 3) Gan errors/experiment_status CUOI CUNG, sau khi da validate va lam sach.
master_summary["errors"] = list(RUN_ERRORS)
master_summary["experiment_status"] = dict(EXPERIMENT_STATUS)

_status_failed = sorted(k for k, v in EXPERIMENT_STATUS.items() if str(v).startswith("failed"))
_status_skipped = sorted(k for k, v in EXPERIMENT_STATUS.items() if str(v).startswith("skipped"))
_status_partial = sorted(k for k, v in EXPERIMENT_STATUS.items() if str(v).startswith("ok_partial") or str(v) == "missing_key" or str(v).startswith("blocked"))
_status_not_run = sorted(k for k, v in EXPERIMENT_STATUS.items() if str(v) == "not_run")
overall_status = "ok" if not (_status_failed or _status_skipped or _status_partial or _status_not_run or RUN_ERRORS) else "partial"
master_summary["overall_status"] = overall_status
master_summary["experiment_status_summary"] = {
    "ok": sorted(k for k, v in EXPERIMENT_STATUS.items() if str(v) == "ok"),
    "partial": _status_partial,
    "skipped": _status_skipped,
    "failed": _status_failed,
    "not_run": _status_not_run,
    "n_errors": len(RUN_ERRORS),
}

_ext1 = (val_to_test_results or {}).get("test_metrics")
_exp1_test_miou = _metric_from(_ext1, "mIoU_4cls", "mIoU")
_exp2_auc = (pareto_data or {}).get("auc_gain", None)
_exp2_n = (pareto_data or {}).get("num_samples", None)
_exp3_corr = (bio_results or {}).get("correlation_metrics", {}) or {}
_exp3_n = _exp3_corr.get("n_patients", _exp3_corr.get("patient_count", None))
_exp3_r = _exp3_corr.get("vtp_pearson_r", None)
_exp3_n_int = _exp3_n if isinstance(_exp3_n, int) else None
_exp3_r_display = _exp3_r if (_exp3_n_int is not None and _exp3_n_int >= 2) else None
_exp4_miou = _metric_from(luad_results, "mIoU_4cls", "mIoU")
_exp4b_miou = _metric_from(luad_transfer_results, "mIoU_4cls", "mIoU")
_exp5_miou = (roi_metrics or {}).get("mIoU_4cls", None)
_exp5_cov = ((roi_metrics or {}).get("coverage", {}) or {}).get("min_coverage_ratio", None)
_exp5_valid = (roi_metrics or {}).get("metric_valid", None)

if _exp3_r_display is not None:
    _exp3_r_text = _num(_exp3_r_display, "{:.3f}")
else:
    _exp3_r_text = f"N/A (n={_exp3_n if _exp3_n is not None else 'N/A'})"

_table_rows = [
    ("EXP1 Val-to-Test", EXPERIMENT_STATUS["exp1_val_to_test"], f"Test mIoU {_pct(_exp1_test_miou)}"),
    ("EXP2 Multi-Mag/Pareto", EXPERIMENT_STATUS["exp2_multimag_pareto"], f"auc_gain {_num(_exp2_auc)} | n={_exp2_n if _exp2_n is not None else 'N/A'}"),
    ("EXP3 Biomarker", EXPERIMENT_STATUS["exp3_biomarker"], f"VTP r {_exp3_r_text} | patients={_exp3_n if _exp3_n is not None else 'N/A'}"),
    ("EXP4 LUAD in-domain", EXPERIMENT_STATUS["exp4_luad_in_domain"], f"mIoU {_pct(_exp4_miou)}"),
    ("EXP4b BCSS->LUAD transfer", EXPERIMENT_STATUS["exp4b_bcss_to_luad_transfer"], f"mIoU {_pct(_exp4b_miou)}"),
    ("EXP5 Whole-ROI", EXPERIMENT_STATUS["exp5_whole_roi"], f"mIoU {_pct(_exp5_miou)} | min tissue cov {_pct(None if _exp5_cov is None else float(_exp5_cov) * 100.0)} | valid={_exp5_valid}"),
]

print("=" * 100)
_tag = "[PARTIAL]" if overall_status != "ok" else "[OK]"
print(f"{_tag} BANG TOM TAT 5 THI NGHIEM (mode={RUN_MODE}, commit={RUN_META.get('git_commit')})")
print("-" * 100)
print(f"{'Experiment'.ljust(28)} | {'Status'.ljust(22)} | Metric")
print("-" * 100)
for _row_name, _row_status, _row_metric in _table_rows:
    print(f"{_row_name.ljust(28)} | {str(_row_status).ljust(22)} | {_row_metric}")
print("-" * 100)
if RUN_ERRORS:
    print(f"[PARTIAL] {len(RUN_ERRORS)} loi/van de duoc ghi trong summary['errors']:")
    for _err in RUN_ERRORS:
        print(f"  - {_err.get('experiment')}/{_err.get('stage')}: {_err.get('type')}: {str(_err.get('message'))[:200]}")
print("GIOI HAN (limitations):")
for _lim in limitations:
    print(f"  - {_lim}")
print("=" * 100)

summary_json_path = OUTPUT_DIR / "scientific_benchmark_summary.json"
summary_json_path.parent.mkdir(parents=True, exist_ok=True)
_summary_ok = False
try:
    # sanitize_for_json chi con xu ly type (numpy...); NaN/Inf da bi nan_to_none loai truoc do.
    _summary_text = json.dumps(sanitize_for_json(master_summary), indent=2, ensure_ascii=False, allow_nan=False)
    json.loads(_summary_text, parse_constant=_reject_constant)
    write_json_atomic(summary_json_path, _summary_text)
except (TypeError, ValueError, OSError) as _sum_err:
    record_error("export_summary", _sum_err, "serialize_or_write")
    EXPERIMENT_STATUS["export_summary"] = "failed"
    print("[FAIL] Summary khong serialize/ghi duoc thanh JSON hop le; khong dong goi ZIP.")
else:
    _summary_ok = True
    print(f"[OK] Exported master benchmark summary: {summary_json_path}")

# 4) Chi dong goi ZIP khi summary serialize duoc VA khong con JSON hong/NaN-Inf.
if not _summary_ok:
    print("[FAIL] Khong dong goi ZIP vi summary khong serialize duoc.")
elif _invalid_jsons or _nonfinite:
    print("[FAIL] Khong dong goi ZIP vi con JSON khong hop le hoac truong NaN/Inf.")
else:
    zip_name = f"SE374_Scientific_Research_Results_{RUN_MODE}_{RUN_TIMESTAMP}.zip"
    zip_out = WORK_ROOT / zip_name
    if zip_out.is_file():
        zip_out.unlink()

    print(f"--> Dang dong goi ket qua nghien cuu tu {OUTPUT_DIR} vao file ZIP...")
    try:
        with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as zf:
            for root, _, files in os.walk(OUTPUT_DIR):
                for f in files:
                    p = Path(root) / f
                    zf.write(p, arcname=p.relative_to(OUTPUT_DIR))
    except (OSError, zipfile.BadZipFile) as _zip_err:
        record_error("export_zip", _zip_err, "zip")
        print(f"[FAIL] Khong tao duoc ZIP: {_zip_err}")
    else:
        zip_size_mb = zip_out.stat().st_size / (1024 * 1024)
        print(f"[OK] Da tao thanh cong goi ket qua: {zip_out.resolve()} ({zip_size_mb:.2f} MB)")

    if "google.colab" in sys.modules:
        try:
            from google.colab import files
            print("--> Dang kich hoat hop thoai tai ve tren trinh duyet cua ban...")
            files.download(str(zip_out))
            print("[THANH CONG] Trinh duyet dang tai tap tin ket qua ve may tinh cua ban!")
        except Exception as err:
            print(f"[HUONG DAN] Neu trinh duyet chan popup tu dong, ban co the mo thanh quan ly file ben trai Colab va tai file '{zip_name}'. Chi tiet: {err}")
    else:
        print(f"[OK] Ket qua duoc luu tai thu muc cuc bo: {zip_out.resolve()}")